In [1]:
import rootutils

root_dir = rootutils.setup_root(".",
                                indicator=".project-root",
                                pythonpath=True)

import pandas as pd
import json
from rdkit import Chem
import os

In [2]:
def all_correct_eval(gt, gen) -> bool:
    gt_splits = gt.split('.')
    gen_splits = gen.split('.')
    gt_mols = [Chem.CanonSmiles(mol) for mol in gt_splits]
    gen_mols = [Chem.CanonSmiles(mol) for mol in gen_splits]
    if all([gen_mol in gt_mols for gen_mol in gen_mols]):
        return 1
    else:
        return 0


def any_correct_eval(gt, gen) -> bool:
    gt_splits = gt.split('.')
    gen_splits = gen.split('.')
    gt_mols = [Chem.CanonSmiles(mol) for mol in gt_splits]
    gen_mols = [Chem.CanonSmiles(mol) for mol in gen_splits]
    if any([gen_mol in gt_mols for gen_mol in gen_mols]):
        return 1
    else:
        return 0


def new_eval(df_results):
    df_results['all_correct'] = df_results.apply(
        lambda row: all_correct_eval(row['output_gt'], row['output_gen']),
        axis=1)
    df_results['any_correct'] = df_results.apply(
        lambda row: any_correct_eval(row['output_gt'], row['output_gen']),
        axis=1)
    all_correct = df_results.all_correct.sum()
    any_correct = df_results.any_correct.sum()
    return all_correct, any_correct, df_results


def get_dataset(path, canonicalize=True):
    df = pd.read_csv(path)
    # canonicalize the input
    if canonicalize:
        df['input'] = df['input'].apply(Chem.CanonSmiles)
    df['input'] = df['input'].astype(str)
    # rename the output column to output_gt
    df.rename(columns={'output': 'output_gt'}, inplace=True)
    return df


def get_results_csv(path, canonicalize=True):
    df = pd.read_csv(path)
    if canonicalize:
        df['input'] = df['input'].apply(Chem.CanonSmiles)
    df['input'] = df['input'].astype(str)
    # rename the output column to output_gen
    df.rename(columns={'output': 'output_gen'}, inplace=True)
    return df


In [3]:
def merge_results(df_dataset, df_results):
    # merge the results_df with the df based on the input column
    df = pd.merge(df_dataset, df_results, on='input', how='inner')
    return df

In [4]:
df_dataset = get_dataset("../benchmarks/sampled_molecules.csv")

In [5]:
df_results = get_results_csv(
    '/home/ubuntu/recursiveLLM/results/benchmarking/ASKCOS/askcos_retro_outcomes.csv'
)

In [6]:
merged_df = merge_results(df_dataset, df_results)

In [7]:
all_correct, any_correct, df_results = new_eval(merged_df)

In [8]:
print('all_correct', all_correct)
print('any_correct', any_correct)


all_correct 109
any_correct 130


In [9]:
def generate_results_csv(folder_path):
    files = [
        f for f in os.listdir(folder_path)
        if os.path.isfile(os.path.join(folder_path, f))
    ]

    # load up the results in a df
    results = {}
    for file in files:
        with open(os.path.join(folder_path, file)) as f:
            file_json = json.load(f)
            try:
                output_gen, reagents, input_smiles = get_results_in_file(
                    file_json)
                results[file.split(".js")[0]] = {
                    "output_gen": output_gen,
                    "reagents": reagents,
                    "input": input_smiles
                }
            except:
                print(file)
    df_results = pd.DataFrame(results).T
    return df_results


def get_results_in_file(file):
    step_1 = file['steps'][0]

    reactants = []
    for react in step_1['reactants']:
        reactants.append(react['smiles'])
    reactants = ".".join(reactants)
    reagents = []
    for reagent in step_1['reagents']:
        reagents.append(reagent['smiles'])
    reagents = ".".join(reagents)
    products = []
    for prod in step_1['products']:
        products.append(prod['smiles'])
    products = ".".join(products)
    return reactants, reagents, products

In [10]:
df_new = generate_results_csv(
    "/home/ubuntu/recursiveLLM/results/benchmarking/AZ_paper/USPTO/")

merged_df2 = merge_results(df_dataset, df_new)

all_correct, any_correct, df_results = new_eval(merged_df2)
print("--------------------------------")
print(all_correct, any_correct)

112.json
121.json
25.json
103.json
67.json
211.json


--------------------------------
73 102
